# Phase 2: build the dataset from Google Drive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhavilkawdia5-ux/leaf_disease_detection/blob/main/ml/notebooks/phase2_dataset.ipynb)

This runs `ml/src/dataset.py` (Phase 2 in `docs/SYSTEM_DESIGN.md`) on the raw datasets
in your Google Drive. It:

1. downloads the public datasets into Drive (first time only),
2. merges all public datasets into the KrishiCare classes from `ml/classes.yaml`,
3. splits them train / val / test 70 / 15 / 15 (the test split is never trained on),
4. prints image counts per class and per split,
5. makes one sheet of 25 sample photos per class, so you can check the labels by eye,
6. saves `processed.zip` and `phase2_counts.txt` to Drive for Phase 3.

Step 3 downloads the public datasets into `MyDrive/leafsight/data/raw/` the first
time. Your own photos, if any, go in `raw/own/<class id>/`. No GPU is needed. Run each cell with **Shift + Enter**, top to bottom.

## 1. Connect Google Drive and get the code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
if os.path.isdir("/content/leaf_disease_detection"):
    !git -C /content/leaf_disease_detection pull -q
else:
    !git clone -q https://github.com/bhavilkawdia5-ux/leaf_disease_detection.git /content/leaf_disease_detection
%cd /content/leaf_disease_detection
!git log --oneline -1

## 2. Install packages

In [ ]:
!pip install -q pyyaml pyarrow pillow

## 3. Download the datasets into Drive

Downloads the six public datasets (`ml/src/download.py`, about 2 GB) to this Colab
machine, then saves one zip per dataset in `MyDrive/leafsight/data/raw/<dataset>/`.
Datasets already in Drive are skipped, so the next time this takes seconds.
The first time it takes about 15 to 30 minutes. If a dataset says FAILED, run the cell again.

In [ ]:
from pathlib import Path
import shutil

DRIVE_DATA = Path("/content/drive/MyDrive/leafsight/data")
DRIVE_RAW = DRIVE_DATA / "raw"
DRIVE_RAW.mkdir(parents=True, exist_ok=True)
DL = Path("/content/dl")
SOURCES = ["plantvillage", "plantdoc", "cold_onion", "ngld_grape", "mendeley_tomato", "tom2024"]

missing = [s for s in SOURCES if not (DRIVE_RAW / s).is_dir() or not any((DRIVE_RAW / s).iterdir())]
print("Already in Drive:", [s for s in SOURCES if s not in missing])
print("To download:", missing)
if missing:
    names = " ".join(missing)
    !python ml/src/download.py --data {DL} --only {names}

for s in missing:
    src = DL / "raw" / s
    if not (src / ".done").exists():       # written by download.py when a dataset is complete
        print(s, ": not finished, run this cell again")
        continue
    files = [p for p in src.rglob("*") if p.is_file() and p.name != ".done"]
    (DRIVE_RAW / s).mkdir(exist_ok=True)
    if len(files) == 1:                    # a single .zip or .parquet: copy as is
        shutil.copy(files[0], DRIVE_RAW / s / files[0].name)
    else:                                  # many images: one zip, read in place by dataset.py
        shutil.make_archive(str(DRIVE_RAW / s / s), "zip", src)
    print(s, ": saved to Drive")

for p in sorted(DRIVE_RAW.iterdir()):
    print(p.name, [f.name for f in p.iterdir()])

## 4. Merge and count

The script reads raw data from Drive but writes the processed images to this Colab
machine first (much faster than writing thousands of small files to Drive).
It takes about 5 to 15 minutes. Read the **Notes** at the end of the output.

In [ ]:
WORK = Path("/content/data")
WORK.mkdir(exist_ok=True)
for name in ("raw", "field_test"):          # read from Drive, never changed
    link, target = WORK / name, DRIVE_DATA / name
    if target.is_dir() and not link.exists():
        link.symlink_to(target)

!python ml/src/dataset.py --data /content/data --preview | tee /content/phase2_counts.txt

## 5. Save the results to Drive

Phase 3 trains from `processed.zip`. `phase2_counts.txt` is the count table above.

In [ ]:
import shutil

shutil.copy("/content/phase2_counts.txt", DRIVE_DATA / "phase2_counts.txt")
zip_path = shutil.make_archive("/content/processed", "zip", WORK / "processed")
shutil.copy(zip_path, DRIVE_DATA / "processed.zip")
print("Saved to", DRIVE_DATA, ":", round(os.path.getsize(zip_path) / 1e6), "MB zip")

## 6. Check the labels by eye

Each sheet shows 25 photos from one class. For each sheet, look for:

- a photo that clearly shows a different disease, or a healthy leaf in a disease class
- something that is not a leaf of that crop (fruit, a hand, another plant, a collage)
- the same photo many times

Write down the class name and what looks wrong. Until an expert is available, this
replaces the expert label check (`docs/datasets.md`, "Workflow without field visits").

In [ ]:
from IPython.display import Image as Show, display

for sheet in sorted((WORK / "processed" / "preview").glob("*.jpg")):
    print(sheet.stem)
    display(Show(filename=str(sheet), width=480))

## 7. What to do next

1. Copy all the text printed by step 4 (or open `MyDrive/leafsight/data/phase2_counts.txt`).
2. Paste it to Claude Code with your notes from step 6, and ask it to update
   `docs/datasets.md` with the new counts.
3. Then tick the Phase 2 "Done when" checks.